# Portfolio VaR and Stress Testing

The same results `main.py` prints, with commentary alongside each one.

This notebook imports the modules and repeats no logic. Every calculation
lives in `data.py`, `risk.py`, and `scenarios.py`.

Run all cells.

In [ ]:
import numpy as np
import pandas as pd

import config, main, plots, risk
import scenarios as sc
from data import fetch_prices, compute_returns, portfolio_returns, weights_array

pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

## 1. The data

Daily adjusted close from Yahoo Finance, cached in `data/prices.csv`.

A return is `(price_today / price_yesterday) - 1`. The first day has no prior
day, so 1,255 prices give 1,254 returns.

In [ ]:
prices  = fetch_prices(config.TICKERS, config.START_DATE, config.END_DATE,
                       cache_path=config.PRICE_CACHE)
returns = compute_returns(prices)

print(f'{len(prices):,} prices -> {len(returns):,} returns')
print(f'{prices.index.min():%Y-%m-%d} to {prices.index.max():%Y-%m-%d}')
prices.tail(3)

## 2. Market statistics

**Volatility** is the standard deviation of returns. **Correlation** measures
how each pair moves together.

In [ ]:
stats = pd.DataFrame({'volatility %': returns.std()*100,
                      'mean return %': returns.mean()*100})
display(stats)
display(returns.corr())

## 3. Portfolio aggregation and diversification

The portfolio return is the weighted sum of the stock returns. This is exact
for simple returns, which is why the project does not use log returns.

The portfolio is **calmer than the weighted average of its parts**. That gap
is diversification, and it comes from the pair terms in `w' Sigma w`.

In [ ]:
w   = weights_array(config.WEIGHTS, config.TICKERS)
pr  = portfolio_returns(returns, config.WEIGHTS)
cov = risk.covariance_matrix(returns)
vol = risk.portfolio_volatility(cov, w)

naive = float(w @ returns.std().to_numpy())
print(f'weighted-average volatility  {naive*100:.4f} %   <- the wrong answer')
print(f'true portfolio volatility    {vol*100:.4f} %')
print(f'diversification gain         {(naive-vol)*100:.4f} % lower')

# Proof: with every correlation at 1.0, the naive answer becomes correct.
vols = sc.volatilities(cov)
perfect = sc.rebuild_covariance(vols, pd.DataFrame(np.ones((3,3)), index=cov.index, columns=cov.columns))
print(f'\nvolatility if all correlations = 1.0: {risk.portfolio_volatility(perfect, w)*100:.4f} %')

## 4. Part A — Value at Risk

> With 95% confidence, the 1-day loss will not exceed the figure below.
> Equivalently: on the worst 5% of days, the loss is larger than it.

**Expected Shortfall** is the average loss across those worst days. It answers
the question VaR leaves open, and it is always the larger number.

In [ ]:
results = main.build_results(returns)
results['results']

### The correctness check

Monte Carlo and parametric share the same covariance matrix and the same
normal assumption. One solves a formula; the other rolls 100,000 dice.
**They must agree.**

The assignment specifies Method 2 without a mean and Method 3 *with* a mean
vector, so they differ slightly by design. Zeroing the Monte Carlo mean
removes that difference.

In [ ]:
for c in config.CONFIDENCE_LEVELS:
    p = results['results'].loc['Parametric', f'VaR {c:.0%}']
    z = results['mc_zero_mean_var'][c]
    print(f'{c:.0%}  parametric ${p:>9,.0f}   MC zero-mean ${z:>9,.0f}   gap {abs(z-p)/p:.2%}')

drift = float(w @ returns.mean().to_numpy()) * config.PORTFOLIO_VALUE
mc_sample = results['results'].loc['Monte Carlo', 'VaR 95%']
print(f'\nportfolio mean daily return is ${drift:,.0f} on ${config.PORTFOLIO_VALUE:,.0f}')
print(f'and that is exactly the gap between the two Monte Carlo variants:')
print(f'  ${results["mc_zero_mean_var"][0.95]:,.0f} - ${mc_sample:,.0f} = ${results["mc_zero_mean_var"][0.95]-mc_sample:,.0f}')

### Fat tails

At 95% the three methods agree closely. At 99% they separate. The normal
curve does not allow for how often extreme days really happen.

**The normal assumption is fine in the middle and wrong in the tail — exactly
where a risk number has to work.**

In [ ]:
r = results['results']
for measure in ['VaR 99%', 'ES 99%']:
    h, p = r.loc['Historical', measure], r.loc['Parametric', measure]
    print(f'{measure}:  historical ${h:>9,.0f}   parametric ${p:>9,.0f}   '
          f'short by ${h-p:>7,.0f}  ({h/p-1:.1%})')

In [ ]:
_ = plots.plot_return_distribution(pr,
        {c: r.loc['Historical', f'VaR {c:.0%}'] for c in config.CONFIDENCE_LEVELS},
        config.PORTFOLIO_VALUE, config.PLOTS_DIR / '1_return_distribution.png')
from IPython.display import Image, display
display(Image(str(config.PLOTS_DIR / '1_return_distribution.png')))

In [ ]:
_ = plots.plot_method_comparison(r, config.PLOTS_DIR / '2_method_comparison.png')
display(Image(str(config.PLOTS_DIR / '2_method_comparison.png')))

## 5. Part B — Stress tests

A stress test is not statistical. VaR reads history and reports a probability.
A stress test asks a direct question with no probability: *this happens, what
do I lose?*

That matters because VaR cannot warn about an event outside its data window.
This window starts in 2021, so it holds no 2008 and no COVID crash.

In [ ]:
for s in results['scenarios']:
    print(f'{s.name:<28} {s.portfolio_return:>8.2%}   loss ${s.loss:>10,.0f}')

worst = r.loc['Parametric', 'VaR 99%']
print(f'\nfor comparison, 99% VaR is ${worst:,.0f}')
for s in results['scenarios']:
    print(f'  {s.name:<28} is {s.loss/worst:.1f}x the 99% VaR')

### Scenario 3 — correlation spike

Volatilities are held fixed. Every pairwise correlation is forced to 0.85.

**No stock becomes riskier.** Risk rises purely because the stocks begin
moving together. In a crisis, correlations run toward 1 and diversification
stops working — precisely when it is needed.

In [ ]:
base_vols     = sc.volatilities(cov)
stressed_cov  = sc.spike_correlations(cov, config.STRESSED_CORRELATION)
stressed_vols = sc.volatilities(stressed_cov)

print('per-stock volatility unchanged:', np.allclose(base_vols, stressed_vols))
print(f'  base     {np.round(base_vols*100, 4)}')
print(f'  stressed {np.round(stressed_vols*100, 4)}')
print(f'\nportfolio volatility  {vol*100:.4f} %  ->  {results["stressed_volatility"]*100:.4f} %\n')

for c in config.CONFIDENCE_LEVELS:
    b, s_ = r.loc['Parametric', f'VaR {c:.0%}'], results['stressed_var'][c]
    print(f'parametric VaR {c:.0%}   ${b:>9,.0f}  ->  ${s_:>9,.0f}   (+{(s_/b-1)*100:.1f} %)')

print('\nThe increase is the same at both levels because parametric VaR is')
print('linear in volatility, so z cancels in the ratio:')
print(f'  {results["stressed_volatility"]*100:.4f} / {vol*100:.4f} = {results["stressed_volatility"]/vol:.3f}')

In [ ]:
_ = plots.plot_correlation_shift(results['base_correlation'], results['stressed_correlation'],
                                 config.PLOTS_DIR / '3_correlation_spike.png')
display(Image(str(config.PLOTS_DIR / '3_correlation_spike.png')))

In [ ]:
_ = plots.plot_scenarios_vs_var(
        {c: r.loc['Parametric', f'VaR {c:.0%}'] for c in config.CONFIDENCE_LEVELS},
        {s.name: s.loss for s in results['scenarios']},
        config.PLOTS_DIR / '4_scenarios_vs_var.png')
display(Image(str(config.PLOTS_DIR / '4_scenarios_vs_var.png')))

## 6. Assumptions and limitations

| Assumption | Applies to | Why it may be wrong |
|---|---|---|
| Returns are normal | Parametric, Monte Carlo | Real returns have fat tails. The 99% loss is understated. |
| The past predicts the future | Historical | This window holds no 2008 and no COVID crash. |
| Mean daily return is zero | Parametric | A deliberate simplification, per the assignment's formula. |
| Correlations are stable | Parametric, Monte Carlo | They are not. Scenario 3 quantifies the effect. |
| Weights stay fixed | All | A real portfolio drifts unless rebalanced. |
| One day, position can be exited | All | Says nothing about a week-long slide. |
| Every past day counts equally | Historical | A quiet day gets the same weight as a panic day. |
| The shock sizes are right | Scenarios 1-3 | Chosen by judgment. A stress test gives the consequence of an assumption, never its odds. |